# 전처리 (Preprocessing)

raw train 데이터를 불러와 EDA에서 확인한 핵심 파생변수만 추가하고, 전처리된 train을 CSV로 저장합니다.

흐름:
1. 경로 설정 및 라이브러리 로드
2. train 데이터 로드 및 기본 확인
3. 파생변수 생성 함수 정의
4. 전처리 실행
5. CSV 저장

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 200)

# ── 경로 설정 (본인 환경에 맞게 수정) ──────────────────────────
DATA_DIR   = Path("../data")
TRAIN_PATH = DATA_DIR / "cc_fraud_train.csv"
TEST_PATH  = DATA_DIR / "cc_fraud_test.csv"
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# train 원본 데이터 로드 및 기본 확인
train = pd.read_csv(TRAIN_PATH)

# pandas가 자동 생성하는 Unnamed 인덱스 컬럼 제거
train = train.loc[:, ~train.columns.str.contains(r"^Unnamed")]

print(f"train shape : {train.shape}")
display(train.head())

# 결측치 확인
missing = train.isna().mean().sort_values(ascending=False)
print("\n결측치 비율 (train)")
display(missing[missing > 0].head(20) if missing.any() else "결측치 없음")

# 타겟 분포 확인
print("\n타겟 분포 (train)")
display(train["is_fraud"].value_counts(normalize=True).rename("ratio").to_frame())


train shape : (1296675, 22)


,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0



결측치 비율 (train)


'결측치 없음'


타겟 분포 (train)


,ratio
is_fraud,
0,0.994211
1,0.005789


In [4]:
# train 원본 데이터 로드 및 기본 확인
test = pd.read_csv(TEST_PATH)

# pandas가 자동 생성하는 Unnamed 인덱스 컬럼 제거
test = test.loc[:, ~test.columns.str.contains(r"^Unnamed")]

print(f"test shape : {test.shape}")
display(test.head())

# 결측치 확인
missing = test.isna().mean().sort_values(ascending=False)
print("\n결측치 비율 (test)")
display(missing[missing > 0].head(20) if missing.any() else "결측치 없음")

# 타겟 분포 확인
print("\n타겟 분포 (test)")
display(test["is_fraud"].value_counts(normalize=True).rename("ratio").to_frame())

test shape : (555719, 22)


,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,Columbia,SC,29209,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,Altonah,UT,84002,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,Bellmore,NY,11710,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0
3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,Titusville,FL,32780,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0
4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,Falmouth,MI,49632,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0



결측치 비율 (test)


'결측치 없음'


타겟 분포 (test)


,ratio
is_fraud,
0,0.99614
1,0.00386


In [5]:
# ── 파생변수 생성 함수 ──────────────────────────────────────────
def haversine_distance(lat1, lon1, lat2, lon2):
    """카드 소지자 거주지 ~ 가맹점 간 직선 거리 계산 (단위: km)"""
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


def add_features(df):
    """
    EDA에서 확인한 핵심 파생변수만 남기도록 전처리한다.
    train / test 양쪽에 동일하게 적용한다.
    """
    df = df.copy()

    # 날짜 파싱
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
    df["dob"] = pd.to_datetime(df["dob"], errors="coerce")

    # 시간 파생변수
    df["trans_hour"] = df["trans_date_trans_time"].dt.hour
    df["trans_dayofweek"] = df["trans_date_trans_time"].dt.dayofweek  # 0=월요일
    df["trans_month"] = df["trans_date_trans_time"].dt.month
    df["trans_day"] = df["trans_date_trans_time"].dt.day

    # 나이 계산
    df["age"] = df["trans_date_trans_time"].dt.year - df["dob"].dt.year

    # 소지자 거주지 ~ 가맹점 거리
    df["distance_km"] = haversine_distance(
        df["lat"], df["long"], df["merch_lat"], df["merch_long"]
    )

    # 카드별 개인 패턴 통계 생성
    card_stats = df.groupby("cc_num").agg(
        card_avg_amt=("amt", "mean"),
        card_std_amt=("amt", "std"),
        card_avg_dist=("distance_km", "mean"),
        card_std_dist=("distance_km", "std"),
        card_avg_hour=("trans_hour", "mean"),
    ).reset_index()

    # 표준편차가 0 또는 결측이면 z-score 계산이 깨지므로 1로 보정
    card_stats["card_std_amt"] = card_stats["card_std_amt"].fillna(1).replace(0, 1)
    card_stats["card_std_dist"] = card_stats["card_std_dist"].fillna(1).replace(0, 1)

    df = df.merge(card_stats, on="cc_num", how="left")

    # EDA 기준으로 최종 유지할 개인 패턴 변수만 생성
    df["amt_zscore"] = (df["amt"] - df["card_avg_amt"]) / df["card_std_amt"]
    df["hour_dev"] = (df["trans_hour"] - df["card_avg_hour"]).abs()
    df["dist_zscore"] = (df["distance_km"] - df["card_avg_dist"]) / df["card_std_dist"]
    df["high_amt_far"] = ((df["amt_zscore"] > 2) & (df["dist_zscore"] > 1)).astype(int)

    # 원본 날짜·개인정보 컬럼 제거 (모델 입력 불필요)
    drop_cols = [
        "trans_date_trans_time", "dob",
        "first", "last", "street", "city",
        "job", "trans_num", "cc_num",
        "card_avg_amt", "card_std_amt", "card_avg_dist", "card_std_dist", "card_avg_hour",
        "dist_zscore",
    ]
    drop_cols = [c for c in drop_cols if c in df.columns]
    return df.drop(columns=drop_cols)

In [6]:
# ── train 전처리 실행 ─────────────────────────────────────────────────
train_fe = add_features(train)

print(f"전처리 후 train shape : {train_fe.shape}")
display(train_fe.head())
display(train_fe.dtypes.to_frame("dtype"))


전처리 후 train shape : (1296675, 22)


,merchant,category,amt,gender,state,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,is_fraud,trans_hour,trans_dayofweek,trans_month,trans_day,age,distance_km,amt_zscore,hour_dev,high_amt_far
0,"fraud_Rippin, Kub and Mann",misc_net,4.97,F,NC,28654,36.0788,-81.1781,3495,1325376018,36.011293,-82.048315,0,0,1,1,1,31,78.597568,-0.651072,14.021203,0
1,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,F,WA,99160,48.8878,-118.2105,149,1325376044,49.159047,-118.186462,0,0,1,1,1,41,30.212176,0.450243,14.021452,0
2,fraud_Lind-Buckridge,entertainment,220.11,M,ID,83252,42.1808,-112.2620,4154,1325376051,43.150704,-112.154481,0,0,1,1,1,57,108.206083,1.518323,10.339960,0
3,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,M,MT,59632,46.2306,-112.1138,1939,1325376076,47.034331,-112.561071,0,0,1,1,1,52,95.673231,-0.186931,12.265720,0
4,fraud_Keeling-Crist,misc_pos,41.96,M,VA,24433,38.4207,-79.4629,99,1325376186,38.674999,-78.632459,0,0,1,1,1,33,77.556744,-0.597057,12.110560,0


,dtype
merchant,str
category,str
amt,float64
gender,str
state,str
zip,int64
lat,float64
long,float64
city_pop,int64
unix_time,int64


In [7]:
# ── test 전처리 실행 ─────────────────────────────────────────────────
test_fe = add_features(test)

print(f"전처리 후 test shape : {test_fe.shape}")
display(test_fe.head())
display(test_fe.dtypes.to_frame("dtype"))

전처리 후 test shape : (555719, 22)


,merchant,category,amt,gender,state,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,is_fraud,trans_hour,trans_dayofweek,trans_month,trans_day,age,distance_km,amt_zscore,hour_dev,high_amt_far
0,fraud_Kirlin and Sons,personal_care,2.86,M,SC,29209,33.9659,-80.9355,333497,1371816865,33.986391,-81.200714,0,12,6,6,21,52,24.561462,-0.372001,1.014062,0
1,fraud_Sporer-Keebler,personal_care,29.84,F,UT,84002,40.3207,-110.4360,302,1371816873,39.450498,-109.960431,0,12,6,6,21,30,104.925092,-0.232006,1.823178,0
2,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,F,NY,11710,40.6729,-73.5365,34496,1371816893,40.495810,-74.196111,0,12,6,6,21,50,59.080078,-0.402674,2.029823,0
3,fraud_Haley Group,misc_pos,60.05,M,FL,32780,28.5697,-80.8191,54767,1371816915,28.812398,-80.883061,0,12,6,6,21,33,27.698567,0.007763,1.680241,0
4,fraud_Johnston-Casper,travel,3.19,M,MI,49632,44.2529,-85.0170,1126,1371816917,44.959148,-85.884734,0,12,6,6,21,65,104.335106,-0.683890,0.319865,0


,dtype
merchant,str
category,str
amt,float64
gender,str
state,str
zip,int64
lat,float64
long,float64
city_pop,int64
unix_time,int64


In [8]:
# ── CSV 저장 ────────────────────────────────────────────────────
# train
train_out = OUTPUT_DIR / "cc_fraud_train_processed.csv"
train_fe.to_csv(train_out, index=False)

# test
test_out = OUTPUT_DIR / "cc_fraud_test_processed.csv"
test_fe.to_csv(test_out, index=False)

print(f"train 저장 완료 → {train_out.resolve()}")
print(f"test 저장 완료 → {test_out.resolve()}")

train 저장 완료 → D:\Lecture\03_Bootcamp\Teamproject\dataset\cc_fraud_train_processed.csv
test 저장 완료 → D:\Lecture\03_Bootcamp\Teamproject\dataset\cc_fraud_test_processed.csv
